# Clean Data from Connectors

Level 1, Python Fundamentals in GenAI Context.

**Problems in this notebook**

- Problem 1: Deduplicate Webhook Events
- Problem 2: Normalize Messy CRM Records

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `01_clean_connector_data_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 1: Deduplicate Webhook Events

**The situation**

A CRM connector sends a webhook every time a contact changes. The provider retries when it does not hear back fast enough, so the same event can arrive two or three times. If the AI workflow handles every copy, the customer gets duplicate follow-up emails and you pay for duplicate model calls.

**Your task**

Write `dedupe_events(events)`. Keep the first copy of each `event_id`, in the order the events arrived. Skip any event whose `event_id` is missing or blank.

**Example**

`[{"event_id": "e1"}, {"event_id": "e1"}, {"event_id": "e2"}]` -> `[{"event_id": "e1"}, {"event_id": "e2"}]`

**Ask before coding**

- If two copies share an id but have different payloads, does the first or the latest win?
- Should an event with no id be skipped, or should the whole batch fail?
- Is this one batch in memory, or must we remember ids across batches and servers?

**Plan to say aloud**

1. Keep a set of ids I have already seen.
2. Walk the events in arrival order.
3. Skip events with a missing or blank id.
4. Skip events whose id is already in the set.
5. Otherwise, add the id to the set and keep the event.

### Prompt for your AI assistant

```text
Write a Python function `dedupe_events(events: list[dict]) -> list[dict]`. Keep only the first event for each `event_id` and preserve the original order. Skip events where `event_id` is missing, not a string, or blank. Do not sort. Standard library only. Show only the function.
```

**Check the AI's code for**

- Uses a dict keyed by id, so the last copy wins instead of the first.
- Sorts the output or builds it from a set, so arrival order is lost.
- Uses `event['event_id']` and crashes with KeyError, or lets a missing id through as `None`.
- Treats a blank string id like `"  "` as valid.

In [ ]:
def dedupe_events(events: list[dict]) -> list[dict]:
    """Return the first event for each event_id, in arrival order.

    Rules:
    - Keep the first copy of each event_id. Drop later copies.
    - Skip events whose event_id is missing, not a string, or blank.
    - Do not reorder the events.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"event_id": "e1"}, {"event_id": "e1"}, {"event_id": "e2"}],), [{"event_id": "e1"}, {"event_id": "e2"}]),
    (([{"event_id": "e1", "v": 1}, {"event_id": "e1", "v": 2}],), [{"event_id": "e1", "v": 1}]),  # first copy wins
    (([{"payload": 1}, {"event_id": "  "}, {"event_id": "e1"}],), [{"event_id": "e1"}]),  # missing and blank ids skipped
    (([{"event_id": "e2"}, {"event_id": "e1"}],), [{"event_id": "e2"}, {"event_id": "e1"}]),  # order kept, not sorted
    (([],), []),
]
run_checks(dedupe_events, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def dedupe_events_reference(events: list[dict]) -> list[dict]:
    seen: set[str] = set()
    kept: list[dict] = []
    for event in events:
        event_id = event.get("event_id")
        if not isinstance(event_id, str) or not event_id.strip():
            continue  # no usable id: we cannot prove it is not a duplicate
        if event_id in seen:
            continue
        seen.add(event_id)
        kept.append(event)
    return kept


assert run_checks(dedupe_events_reference, cases)

**Explain it in 30 seconds**

> "Webhook providers retry, so the same event can arrive more than once. I keep a set of ids I have seen and walk the list in order, keeping only the first copy. Events with no id are skipped on purpose, because I cannot prove they are not duplicates. In production the seen set has to live in shared storage so every server agrees."

**Edge cases to mention**

- Same id with a different payload: the first copy wins, and I would log the mismatch.
- Missing or blank id: skipped on purpose, never processed blindly.
- Events that arrive out of order.
- Two tenants that happen to use the same event id.

**If they ask "how would this work in production?"**

- Store processed ids in a shared store with a uniqueness rule (a database unique key or Redis `SET NX`), so all servers agree.
- Make the key `tenant_id + event_id`, and expire old keys after a retention window.
- Count duplicates and skipped events so a sudden spike shows up on a dashboard.

## Problem 2: Normalize Messy CRM Records

**The situation**

A sales assistant pulls contacts from Salesforce, HubSpot and CSV exports. The same field shows up as `Email`, `e-mail` or `Email Address`, often with stray spaces or odd capital letters. The prompt builder needs one clean shape every time.

**Your task**

Write `normalize_crm_record(record)`. Return a dict with exactly four keys: `email`, `name`, `company`, `phone`. Match source keys loosely, trim values, lowercase only the email, and use `None` for anything missing or empty.

**Example**

`{"Email": " JOHN@EXAMPLE.COM ", "Company Name": " Acme Inc "}` -> `{"email": "john@example.com", "name": None, "company": "Acme Inc", "phone": None}`

**Ask before coding**

- Which source field names map to each output field?
- If two aliases are both present with different values, which one wins?
- Should a bad email be fixed, flagged, or passed through as it is?

**Plan to say aloud**

1. Write an alias list for each output field.
2. Build a lookup of the record's keys, trimmed and lowercased.
3. For each output field, take the first alias that has a non-empty value.
4. Trim the value, and lowercase it only if it is the email.
5. Use None when nothing is found.

### Prompt for your AI assistant

```text
Write a Python function `normalize_crm_record(record: dict) -> dict`. Output exactly the keys email, name, company, phone. Match input keys case-insensitively after trimming spaces, using these aliases: email = email, e-mail, email address; name = name, full name, full_name; company = company, company name, account; phone = phone, mobile, phone number. Take the first alias with a non-empty value. Convert values to strings and trim them. Lowercase the email only. Missing or empty values become None. Standard library only. Show only the function.
```

**Check the AI's code for**

- Lowercases every value, so names and company names lose their capitals.
- Matches keys exactly, so `EMAIL` or `Email Address ` is missed.
- Returns an empty string instead of None for blank values.
- Leaves out keys that were not found, so the output shape changes from record to record.

In [ ]:
def normalize_crm_record(record: dict) -> dict:
    """Map a messy CRM record to {"email", "name", "company", "phone"}.

    Rules:
    - Match input keys ignoring case and surrounding spaces.
    - Aliases: email:   email, e-mail, email address
               name:    name, full name, full_name
               company: company, company name, account
               phone:   phone, mobile, phone number
    - Take the first alias with a non-empty value.
    - Convert values to str and trim them. Lowercase the email only.
    - Missing or empty values become None. Always return all four keys.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"Email": " JOHN@EXAMPLE.COM ", "Company Name": " Acme Inc "},),
     {"email": "john@example.com", "name": None, "company": "Acme Inc", "phone": None}),
    (({},), {"email": None, "name": None, "company": None, "phone": None}),
    (({"Full Name": "  Jane Doe ", "Mobile": " +1 555 0100 "},),
     {"email": None, "name": "Jane Doe", "company": None, "phone": "+1 555 0100"}),
    # name keeps its capitals, blank company becomes None
    (({"NAME": "Ana Lopez", "e-mail": "Ana@X.io", "Account": "  "},),
     {"email": "ana@x.io", "name": "Ana Lopez", "company": None, "phone": None}),
    # a phone stored as a number comes out as a string
    (({"phone": 4915112345678},), {"email": None, "name": None, "company": None, "phone": "4915112345678"}),
]
run_checks(normalize_crm_record, cases)

### Reference solution

Try first. Then compare.

In [ ]:
CRM_ALIASES = {
    "email": ["email", "e-mail", "email address"],
    "name": ["name", "full name", "full_name"],
    "company": ["company", "company name", "account"],
    "phone": ["phone", "mobile", "phone number"],
}


def normalize_crm_record_reference(record: dict) -> dict:
    lookup = {str(key).strip().lower(): value for key, value in record.items()}
    out: dict[str, str | None] = {}
    for field, aliases in CRM_ALIASES.items():
        value = None
        for alias in aliases:
            raw = lookup.get(alias)
            if raw is not None and str(raw).strip():
                value = str(raw).strip()
                break
        if field == "email" and value:
            value = value.lower()
        out[field] = value
    return out


assert run_checks(normalize_crm_record_reference, cases)

**Explain it in 30 seconds**

> "Every connector names fields its own way, so I keep a small alias list for each output field. I compare keys after trimming and lowercasing, take the first alias with a real value, and trim it. Only the email gets lowercased, because names and company names need their capitals. Every record comes out with the same four keys, so the prompt builder never has to guess."

**Edge cases to mention**

- Two aliases with different values: the first alias in the list wins, and that rule should be agreed with the customer.
- Non-string values, such as a phone number stored as a number.
- Blank strings that should count as missing.
- An email that is clearly invalid, like `john at acme`.

**If they ask "how would this work in production?"**

- Keep alias maps per connector and per customer in config, not hard-coded in the function.
- Return problems (bad email, conflicting aliases) next to the clean record instead of hiding them.
- Track how often each source fails to normalize, so a broken connector is caught early.

## Wrap-up

- Data from connectors is untrusted: check its shape before you use it.
- Decide on purpose what happens to bad rows (skip, flag, or fail), and say that choice out loud.
- Return a predictable output so the next step in the AI pipeline never has to guess.